# General setup

In [1]:
# Imports
import matplotlib
matplotlib.use('Agg')
import argparse
import re
import json
import warnings
import numpy as np
from modules.CHILI import CHILI
from modules.net import SCVAE
from torch_geometric.loader import DataLoader
import torch
from torch.utils.data import TensorDataset
import datetime
import pathlib
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from ase import Atoms
from ase.io import write
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from modules.loss_functions import weighted_MSELoss, weighted_CrossEntropyLoss
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

In [2]:
# Functions
def create_cif(cell_params, cell_positions, cell_atoms, filename, prediction=True, composition=None, simplified_atom_identities=False):
    """
    Create a CIF file from the cell parameters, positions and atoms
    """
    if prediction:
        # Find argmax of atoms
        cell_atoms = np.argmax(cell_atoms, axis=1)

    # Remove atoms with atom number 0
    cell_positions = cell_positions[cell_atoms != 0]
    cell_atoms = cell_atoms[cell_atoms != 0]
    
    # Remove atoms not in the unit cell
    cell_atoms = cell_atoms[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    cell_positions = cell_positions[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    
    
    if simplified_atom_identities:
        cell_atoms = np.where(cell_atoms == 1, 8, cell_atoms)
        cell_atoms = np.where(cell_atoms == 2, 26, cell_atoms)
    
    # Create Atoms object
    atoms = Atoms(cell_atoms, scaled_positions=cell_positions, cell=cell_params)

    if not composition:
        composition = str(atoms.symbols)

    # Write CIF
    write(filename + f'.cif', images=atoms, format='cif') # _{composition}

    if not prediction:
        return composition
    return None

In [3]:
# Setup
model_path = './models/Supercell_latentLog_beta_annealing_3d_latentMSE_biggerDecoder_v2/' # './models/Supercell_latentLog_beta_annealing_3d_latentMSE_biggerDecoder_v2/'  './models/Interpolation_v2_Unitcell_latentLog_2d_latentMSE_biggerDecoder/'
model_name = 'best_model_annealed.pth' # 'best_model_annealed.pth'    'best_model.pth'
setup_json_path = f'{model_path}setup_json.json'
with open(setup_json_path, 'r') as setup_json_file:
    setup_json = json.load(setup_json_file)

# latent_space_version = '_prior' # '' or '_prior'

# Make prediction folders
experimental_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions'
pathlib.Path(experimental_folder).mkdir(parents=True, exist_ok=True)

interpolation_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions'
pathlib.Path(interpolation_folder).mkdir(parents=True, exist_ok=True)

sample_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions'
pathlib.Path(sample_folder).mkdir(parents=True, exist_ok=True)

# Make paper figures folder
paper_figures_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures'
pathlib.Path(paper_figures_folder).mkdir(parents=True, exist_ok=True)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
legend_order = [
    'RheniumTrioxide', 
    'Wurtzite', 
    'CaesiumChloride', 
    'RockSalt', 
    'NickelArsenide', 
    'Rutile', 
    'ZincBlende', 
    'CadmiumIodide',
    'AntiFluorite',
    'Fluorite',
    'CadmiumChloride',
    'Spinel',
]

In [5]:
# Load model
model = SCVAE(
    latent_dim=setup_json['model']['latent_dim'],
    out_dim=setup_json['model']['out_dim'],
    prior_factor=setup_json['model']['prior_factor'],
    gnn_dim=setup_json['model']['gnn_dim'],
    gnn_heads=setup_json['model']['gnn_heads'],
    gnn_edge_dim=setup_json['model']['gnn_edge_dim'],
    scattering_channels=setup_json['model']['scattering_channels'],
    scattering_dim=setup_json['model']['scattering_dim'],
    scattering_kernel_size=setup_json['model']['scattering_kernel_size'],
    scattering_stride=setup_json['model']['scattering_stride'],
    scattering_padding=setup_json['model']['scattering_padding'],
    composition_dim=setup_json['model']['composition_dim'],
    decoder_hidden_dim=setup_json['model']['decoder_hidden_dim'],
    position_output_dim=setup_json['model']['position_output_dim'],
    atom_output_dim=setup_json['model']['atom_output_dim'],
    cell_output_dim=setup_json['model']['cell_output_dim'],
    seperate_decoder=setup_json['training']['seperate_decoder'],
).to(device)

# Load model weights
model.load_state_dict(torch.load(model_path + model_name))

<All keys matched successfully>

In [6]:
# Load normalization parameters
if setup_json['data']['normalize_cell_parameters']:
    cell_means = torch.tensor([
        setup_json['data']['cell_normalization']['a']['mean'],
        setup_json['data']['cell_normalization']['b']['mean'],
        setup_json['data']['cell_normalization']['c']['mean'],
        setup_json['data']['cell_normalization']['alpha']['mean'],
        setup_json['data']['cell_normalization']['beta']['mean'],
        setup_json['data']['cell_normalization']['gamma']['mean'],
    ]).float().to(device)
    cell_stds = torch.tensor([
        setup_json['data']['cell_normalization']['a']['std'],
        setup_json['data']['cell_normalization']['b']['std'],
        setup_json['data']['cell_normalization']['c']['std'],
        setup_json['data']['cell_normalization']['alpha']['std'],
        setup_json['data']['cell_normalization']['beta']['std'],
        setup_json['data']['cell_normalization']['gamma']['std'],
    ]).float().to(device)

if setup_json['data']['normalize_atom_positions']:
    atom_position_means = torch.tensor(setup_json['data']['atom_position_normalization']['mean']).float().to(device)
    atom_position_stds = torch.tensor(setup_json['data']['atom_position_normalization']['std']).float().to(device)

if setup_json['data']['normalize_distances']:
    distance_means = torch.tensor(setup_json['data']['distance_normalization']['mean']).float().to(device)
    distance_stds = torch.tensor(setup_json['data']['distance_normalization']['std']).float().to(device)

beta = setup_json['training']['beta']
out_dim = setup_json['model']['out_dim']

# Load model test results

## Code

In [7]:
# Load results from test script
with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/losses_{model_name[:-4]}.json', 'r') as f:
    losses_json = json.load(f)
df_losses = pd.DataFrame(losses_json)

with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/reconstructions_{model_name[:-4]}.json', 'r') as f:
    rec_json = json.load(f)
df_rec = pd.DataFrame(rec_json)
# Capitalize crystal types if not already
name_map = {'interpolated': 'Interpolated', 'rocksalt': 'RockSalt', 'spinel': 'Spinel', 'zincblende': 'ZincBlende'}
df_rec['crystalType'] = df_rec['crystalType'].apply(lambda x: name_map[x] if x in name_map else x)

# Change column names
df_rec.rename(columns={
    'cell_parameters': 'pred_cell_parameters',
    'cell_atoms': 'pred_cell_atoms',
    'cell_positions': 'pred_cell_positions',
    'cell_parameters_prior': 'pred_cell_parameters_prior',
    'cell_atoms_prior': 'pred_cell_atoms_prior',
    'cell_positions_prior': 'pred_cell_positions_prior',
}, inplace=True)

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    df_rec[['ls1', 'ls2']] = df_rec['latent_space_mean'].apply(pd.Series)
    df_rec[['ls1_prior', 'ls2_prior']] = df_rec['latent_space_mean_prior'].apply(pd.Series)
else:
    # Perform PCA
    pca = PCA(n_components=2)
    pca.fit(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1', 'pc2']] = pca.transform(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_rec['latent_space_mean_prior'].values.tolist()))

In [8]:
df_losses.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,crystalType,particleSize
0,-5.186489,0.005573,0.004005,0.001567,4.257474e-09,0.000018,Rutile,53.859127
1,-6.892604,0.000764,0.000441,0.000322,9.685696e-07,0.000243,RheniumTrioxide,11.723881
2,-7.835576,0.000293,0.000112,0.000180,1.113324e-06,0.000098,RheniumTrioxide,53.645557
3,-4.656991,0.008764,0.003214,0.005518,3.139985e-05,0.000700,Spinel,12.495686
4,-6.759521,0.000673,0.000140,0.000533,8.514949e-09,0.000455,Spinel,33.773544


In [9]:
df_rec.head()

,crystalType,n_atoms,n_oxygens,n_metals,pred_cell_parameters,pred_cell_positions,pred_cell_atoms,pred_cell_parameters_prior,pred_cell_positions_prior,pred_cell_atoms_prior,...,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,Rutile,56,32,24,"[4.533034324645996, 4.518996715545654, 2.86099...","[[0.014750000089406967, -0.014750000089406967,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...","[4.547970771789551, 4.537806510925293, 2.86309...","[[0.009096876718103886, -0.020784586668014526,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",...,"[0.08817844837903976, 0.12891697883605957]","[0.3466842770576477, -1.4527558088302612]","[0.08854163438081741, 0.12905548512935638]","[0.1702643632888794, 0.1702643632888794, -0.89...","[[0.0, 0.0, 0.0], [0.30000001192092896, 0.3000...","[38, 8, 38, 8, 8, 8, 38, 38, 38, 8, 8, 8, 8, 3...",0.348047,-1.452304,0.346684,-1.452756
1,RheniumTrioxide,56,36,20,"[3.873868227005005, 3.8785364627838135, 3.7729...","[[0.0009800000116229057, 0.004410000052303076,...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[3.8862388134002686, 3.892515182495117, 3.7686...","[[0.0021697664633393288, 0.004440604709088802,...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",...,"[0.06011437997221947, 0.08880679309368134]","[-0.32790911197662354, 0.8985519409179688]","[0.0616922527551651, 0.09096308797597885]","[-0.3129006326198578, -0.3129006326198578, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[37, 8, 8, 8, 37, 37, 37, 8, 8, 8, 8, 8, 8, 37...",-0.326368,0.901145,-0.327909,0.898552
2,RheniumTrioxide,56,36,20,"[3.8755974769592285, 3.8816440105438232, 3.865...","[[0.000699999975040555, 0.008729999884963036, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[3.911433458328247, 3.9128384590148926, 4.2709...","[[0.026861630380153656, 0.03532092645764351, 0...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",...,"[0.06620895117521286, 0.09541012346744537]","[-0.32047486305236816, 0.902169942855835]","[0.06682302057743073, 0.09742937237024307]","[-0.3164351284503937, -0.3164351284503937, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[39, 8, 8, 8, 39, 39, 39, 8, 8, 8, 8, 8, 8, 39...",-0.320265,0.904384,-0.320475,0.902170
3,Spinel,56,32,24,"[7.247450828552246, 7.211139678955078, 7.14943...","[[0.4954499900341034, 0.4786899983882904, 0.47...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[7.53020715713501, 7.510119438171387, 7.461524...","[[0.5184663534164429, 0.504451334476471, 0.499...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",...,"[0.05290481820702553, 0.10768686980009079]","[-0.675494909286499, 0.3540700376033783]","[0.0538572333753109, 0.1070767417550087]","[1.7886549234390259, 1.7886549234390259, 0.503...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...",-0.671053,0.347558,-0.675495,0.354070
4,Spinel,56,32,24,"[8.843179702758789, 8.841020584106445, 8.79512...","[[0.5167800188064575, 0.5118200182914734, 0.49...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[9.110713005065918, 9.124397277832031, 9.10257...","[[0.5089696645736694, 0.5091920495033264, 0.51...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",...,"[0.05827934294939041, 0.13212887942790985]","[-0.950782835483551, 0.12682023644447327]","[0.05925476551055908, 0.13336198031902313]","[2.667485475540161, 2.667485475540161, 0.95396...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 5...",-0.947091,0.133730,-0.950783,0.126820


In [10]:
df_combined = pd.concat([df_losses.drop('crystalType', axis=1), df_rec], axis=1)
df_combined.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,crystalType,n_atoms,n_oxygens,...,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,-5.186489,0.005573,0.004005,0.001567,4.257474e-09,0.000018,53.859127,Rutile,56,32,...,"[0.08817844837903976, 0.12891697883605957]","[0.3466842770576477, -1.4527558088302612]","[0.08854163438081741, 0.12905548512935638]","[0.1702643632888794, 0.1702643632888794, -0.89...","[[0.0, 0.0, 0.0], [0.30000001192092896, 0.3000...","[38, 8, 38, 8, 8, 8, 38, 38, 38, 8, 8, 8, 8, 3...",0.348047,-1.452304,0.346684,-1.452756
1,-6.892604,0.000764,0.000441,0.000322,9.685696e-07,0.000243,11.723881,RheniumTrioxide,56,36,...,"[0.06011437997221947, 0.08880679309368134]","[-0.32790911197662354, 0.8985519409179688]","[0.0616922527551651, 0.09096308797597885]","[-0.3129006326198578, -0.3129006326198578, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[37, 8, 8, 8, 37, 37, 37, 8, 8, 8, 8, 8, 8, 37...",-0.326368,0.901145,-0.327909,0.898552
2,-7.835576,0.000293,0.000112,0.000180,1.113324e-06,0.000098,53.645557,RheniumTrioxide,56,36,...,"[0.06620895117521286, 0.09541012346744537]","[-0.32047486305236816, 0.902169942855835]","[0.06682302057743073, 0.09742937237024307]","[-0.3164351284503937, -0.3164351284503937, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[39, 8, 8, 8, 39, 39, 39, 8, 8, 8, 8, 8, 8, 39...",-0.320265,0.904384,-0.320475,0.902170
3,-4.656991,0.008764,0.003214,0.005518,3.139985e-05,0.000700,12.495686,Spinel,56,32,...,"[0.05290481820702553, 0.10768686980009079]","[-0.675494909286499, 0.3540700376033783]","[0.0538572333753109, 0.1070767417550087]","[1.7886549234390259, 1.7886549234390259, 0.503...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...",-0.671053,0.347558,-0.675495,0.354070
4,-6.759521,0.000673,0.000140,0.000533,8.514949e-09,0.000455,33.773544,Spinel,56,32,...,"[0.05827934294939041, 0.13212887942790985]","[-0.950782835483551, 0.12682023644447327]","[0.05925476551055908, 0.13336198031902313]","[2.667485475540161, 2.667485475540161, 0.95396...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 5...",-0.947091,0.133730,-0.950783,0.126820


In [11]:
# Load latent log if it exists
import yaml
try:
    df_latent_log = pd.read_csv(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_log.csv')

    df_latent_log = df_latent_log.drop(df_latent_log[df_latent_log['posterior_mean'] == 'posterior_mean'].index)

    # df_latent_log = df_latent_log[:1000]

    # Select only every tenth epoch
    # Convert epoch to ints
    df_latent_log['epoch'] = df_latent_log['epoch'].astype(np.int16)

    df_latent_log = df_latent_log[df_latent_log['epoch'] % 10 == 0]

    df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']] = df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']].applymap(yaml.safe_load)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_latent_log[['ls1', 'ls2']] = df_latent_log['posterior_mean'].apply(pd.Series)
        df_latent_log[['ls1_prior', 'ls2_prior']] = df_latent_log['prior_mean'].apply(pd.Series)
    else:
        # Perform PCA
        # pca = PCA(n_components=2)
        # pca.fit(np.array(df_latent_log['posterior_mean'].values.tolist()))
        df_latent_log[['pc1', 'pc2']] = pca.transform(np.array(df_latent_log['posterior_mean'].values.tolist()))

        df_latent_log[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_latent_log['prior_mean'].values.tolist()))
        
    print(df_latent_log.columns)
except:
    df_latent_log = None
    print('No latent log found')

Index(['epoch', 'posterior_mean', 'posterior_std', 'prior_mean', 'prior_std',
       'np_size', 'crystal_type', 'crystal_system', 'space_group', 'ls1',
       'ls2', 'ls1_prior', 'ls2_prior'],
      dtype='object')


## Loss statistics

In [12]:
df_combined.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,crystalType,n_atoms,n_oxygens,...,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,-5.186489,0.005573,0.004005,0.001567,4.257474e-09,0.000018,53.859127,Rutile,56,32,...,"[0.08817844837903976, 0.12891697883605957]","[0.3466842770576477, -1.4527558088302612]","[0.08854163438081741, 0.12905548512935638]","[0.1702643632888794, 0.1702643632888794, -0.89...","[[0.0, 0.0, 0.0], [0.30000001192092896, 0.3000...","[38, 8, 38, 8, 8, 8, 38, 38, 38, 8, 8, 8, 8, 3...",0.348047,-1.452304,0.346684,-1.452756
1,-6.892604,0.000764,0.000441,0.000322,9.685696e-07,0.000243,11.723881,RheniumTrioxide,56,36,...,"[0.06011437997221947, 0.08880679309368134]","[-0.32790911197662354, 0.8985519409179688]","[0.0616922527551651, 0.09096308797597885]","[-0.3129006326198578, -0.3129006326198578, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[37, 8, 8, 8, 37, 37, 37, 8, 8, 8, 8, 8, 8, 37...",-0.326368,0.901145,-0.327909,0.898552
2,-7.835576,0.000293,0.000112,0.000180,1.113324e-06,0.000098,53.645557,RheniumTrioxide,56,36,...,"[0.06620895117521286, 0.09541012346744537]","[-0.32047486305236816, 0.902169942855835]","[0.06682302057743073, 0.09742937237024307]","[-0.3164351284503937, -0.3164351284503937, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[39, 8, 8, 8, 39, 39, 39, 8, 8, 8, 8, 8, 8, 39...",-0.320265,0.904384,-0.320475,0.902170
3,-4.656991,0.008764,0.003214,0.005518,3.139985e-05,0.000700,12.495686,Spinel,56,32,...,"[0.05290481820702553, 0.10768686980009079]","[-0.675494909286499, 0.3540700376033783]","[0.0538572333753109, 0.1070767417550087]","[1.7886549234390259, 1.7886549234390259, 0.503...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...",-0.671053,0.347558,-0.675495,0.354070
4,-6.759521,0.000673,0.000140,0.000533,8.514949e-09,0.000455,33.773544,Spinel,56,32,...,"[0.05827934294939041, 0.13212887942790985]","[-0.950782835483551, 0.12682023644447327]","[0.05925476551055908, 0.13336198031902313]","[2.667485475540161, 2.667485475540161, 0.95396...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 5...",-0.947091,0.133730,-0.950783,0.126820


In [13]:
df_combined.columns

Index(['total', 'reconstruction_loss', 'cell_parameters', 'cell_positions',
       'cell_atoms', 'kld', 'particleSize', 'crystalType', 'n_atoms',
       'n_oxygens', 'n_metals', 'pred_cell_parameters', 'pred_cell_positions',
       'pred_cell_atoms', 'pred_cell_parameters_prior',
       'pred_cell_positions_prior', 'pred_cell_atoms_prior',
       'latent_space_mean', 'latent_space_std', 'latent_space_mean_prior',
       'latent_space_std_prior', 'true_cell_parameters', 'true_cell_positions',
       'true_cell_atoms', 'ls1', 'ls2', 'ls1_prior', 'ls2_prior'],
      dtype='object')

In [14]:
df_combined.describe()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,n_atoms,n_oxygens,n_metals,ls1,ls2,ls1_prior,ls2_prior
count,318.000000,318.000000,318.000000,318.000000,3.180000e+02,318.000000,318.000000,318.0,318.000000,318.000000,318.000000,318.000000,318.000000,318.000000
mean,-5.448053,0.008061,0.006500,0.001526,3.459758e-05,0.092932,33.330859,56.0,27.761006,28.238994,0.137736,0.214800,0.136724,0.206137
std,1.503879,0.014270,0.012779,0.006571,3.291256e-04,1.072222,14.406886,0.0,5.986798,5.986798,0.539775,0.888348,0.535900,0.887293
min,-8.179953,0.000169,0.000028,0.000090,0.000000e+00,0.000001,8.903019,56.0,18.000000,19.000000,-1.536427,-1.479403,-1.530814,-1.484257
25%,-6.397877,0.001040,0.000580,0.000284,2.767358e-08,0.000093,22.631171,56.0,21.250000,24.000000,-0.308276,-0.404836,-0.299060,-0.410666
50%,-5.578234,0.002902,0.001953,0.000461,9.887934e-07,0.000344,33.579393,56.0,28.500000,27.500000,0.064326,0.194893,0.067867,0.189320
75%,-4.642443,0.008262,0.006466,0.000945,3.584197e-06,0.001076,43.996708,56.0,32.000000,34.750000,0.540588,0.898915,0.562017,0.897784
max,2.900980,0.108904,0.108365,0.080138,5.385634e-03,17.691074,56.416878,56.0,37.000000,38.000000,1.198118,2.169995,1.193852,2.161826


In [15]:
# Describe for each crystal type
df_combined[['crystalType', 'cell_parameters', 'cell_positions', 'cell_atoms']].groupby('crystalType').describe(percentiles=[0.5])

cell_parameters                                          \
                          count      mean       std       min       50%   
crystalType                                                               
AntiFluorite               26.0  0.018310  0.029477  0.000190  0.004448   
CadmiumChloride            27.0  0.006337  0.009332  0.000172  0.001497   
CadmiumIodide              26.0  0.005932  0.008299  0.000362  0.001954   
CaesiumChloride            27.0  0.004542  0.006340  0.000738  0.001977   
Fluorite                   27.0  0.006437  0.012550  0.000086  0.000603   
NickelArsenide             26.0  0.005980  0.008913  0.000095  0.002145   
RheniumTrioxide            27.0  0.000394  0.000615  0.000028  0.000217   
RockSalt                   27.0  0.004955  0.007051  0.000092  0.002327   
Rutile                     26.0  0.002536  0.001513  0.000149  0.002279   
Spinel                     26.0  0.006845  0.009189  0.000140  0.002855   
Wurtzite                   26.0  0.008403  0.016266  0.000172  0.002265   
ZincBlende                 27.0  0.007660  0.011575  0.000158  0.003138   

                          cell_positions                                \
                      max          count      mean       std       min   
crystalType                                                              
AntiFluorite     0.108365           26.0  0.000375  0.000151  0.000214   
CadmiumChloride  0.035465           27.0  0.000711  0.000335  0.000294   
CadmiumIodide    0.030005           26.0  0.000926  0.003354  0.000173   
CaesiumChloride  0.028845           27.0  0.001993  0.001105  0.001087   
Fluorite         0.051453           27.0  0.000317  0.000398  0.000167   
NickelArsenide   0.041107           26.0  0.001044  0.000885  0.000456   
RheniumTrioxide  0.002597           27.0  0.000321  0.000209  0.000139   
RockSalt         0.028465           27.0  0.001811  0.004361  0.000090   
Rutile           0.006588           26.0  0.006772  0.020565  0.000645   
Spinel           0.040618           26.0  0.000910  0.001301  0.000305   
Wurtzite         0.078158           26.0  0.002383  0.007114  0.000261   
ZincBlende       0.048091           27.0  0.000873  0.001528  0.000195   

                                    cell_atoms                              \
                      50%       max      count          mean           std   
crystalType                                                                  
AntiFluorite     0.000323  0.000723       26.0  5.950958e-06  7.057250e-06   
CadmiumChloride  0.000621  0.001732       27.0  3.961793e-06  8.354876e-06   
CadmiumIodide    0.000220  0.017359       26.0  2.962803e-05  1.484736e-04   
CaesiumChloride  0.001725  0.007183       27.0  8.879402e-07  3.329003e-06   
Fluorite         0.000213  0.002197       27.0  3.220321e-06  1.124607e-05   
NickelArsenide   0.000689  0.003906       26.0  5.161307e-07  1.104557e-06   
RheniumTrioxide  0.000294  0.001300       27.0  3.654095e-06  7.121597e-06   
RockSalt         0.000395  0.017350       27.0  3.175865e-04  1.096095e-03   
Rutile           0.001000  0.080138       26.0  4.756887e-08  1.194179e-07   
Spinel           0.000447  0.005518       26.0  2.212650e-06  7.709738e-06   
Wurtzite         0.000437  0.036803       26.0  2.248682e-05  5.975013e-05   
ZincBlende       0.000376  0.007432       27.0  1.958323e-05  4.830959e-05   

                                                           
                          min           50%           max  
crystalType                                                
AntiFluorite     1.177185e-06  3.737987e-06  3.264668e-05  
CadmiumChloride  9.004531e-07  1.928591e-06  4.430818e-05  
CadmiumIodide    0.000000e+00  8.514948e-09  7.575270e-04  
CaesiumChloride  0.000000e+00  1.490116e-08  1.676041e-05  
Fluorite         1.915863e-08  1.830713e-07  5.857528e-05  
NickelArsenide   2.128737e-09  2.235174e-08  4.402162e-06  
RheniumTrioxide  8.961935e-07  1.175052e-06  3.657536e-05  
RockSalt

In [16]:
def position_MAE(y_pred, y_true):
    """
    Mean absolute error for positions
    """
    return torch.mean(
        torch.sqrt(
            torch.sum(
                torch.nn.functional.mse_loss(
                    y_pred, 
                    y_true, 
                    reduction='none'
                ),
            dim=1)),
        dim=0
    )
    
def calculate_cell_errors(pred_cell_params, gt_cell_params):
    """
    Calculate the cell errors
    """
    # Calculate side length error
    side_length_error = torch.nn.functional.l1_loss(
        torch.tensor(pred_cell_params[:3]),
        torch.tensor(gt_cell_params[:3]),
    ).item()

    # Calculate angle error
    angle_error = torch.nn.functional.l1_loss(
        torch.tensor(pred_cell_params[3:]),
        torch.tensor(gt_cell_params[3:]),
    ).item()

    return side_length_error, angle_error

def calculate_atom_accuracy(pred_atoms, gt_atoms):
    """
    Calculate the accuracy of the predicted atoms
    """
    gt_atoms_simple =  np.where(np.array(gt_atoms) == 8, 1, 2)
    
    return np.sum(np.array(pred_atoms) == gt_atoms_simple) / len(gt_atoms_simple) * 100

In [17]:
# TODO: Calculate positional error in Å
frac_positional_error_list = []
positional_error_list = []
cell_length_error_list = []
cell_angle_error_list = []
atom_accuracy_list = []

prior_frac_positional_error_list = []
prior_positional_error_list = []
prior_cell_length_error_list = []
prior_cell_angle_error_list = []
prior_atom_accuracy_list = []

for index in tqdm(range(len(df_combined))):
    row = df_combined.iloc[index]

    # De-normalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_params = np.array(row['true_cell_parameters'])
        cell_params = (cell_params * cell_stds.cpu().numpy()) + cell_means.cpu().numpy()
    else:
        cell_params = np.array(row['true_cell_parameters'])

    gt_unit_cell = Atoms(
        numbers=np.array(row['true_cell_atoms']),
        scaled_positions=np.array(row['true_cell_positions']),
        cell=cell_params,
        pbc=True,
    )
    gt_abs_positions = gt_unit_cell.get_positions()
    
    pred_unit_cell = Atoms(
        numbers=np.array(row['true_cell_atoms']),
        scaled_positions=np.array(row['pred_cell_positions']),
        cell=cell_params,
        pbc=True,
    )
    pred_abs_positions = pred_unit_cell.get_positions()
    
    prior_pred_unit_cell = Atoms(
        numbers=np.array(row['true_cell_atoms']),
        scaled_positions=np.array(row['pred_cell_positions_prior']),
        cell=cell_params,
        pbc=True,
    )
    prior_pred_abs_positions = prior_pred_unit_cell.get_positions()

    # Calculate positional error
    frac_positional_error = position_MAE(
        torch.tensor(row['pred_cell_positions']),
        torch.tensor(row['true_cell_positions']),
    ).numpy()
    frac_positional_error_list.append(frac_positional_error.item())
    
    prior_frac_positional_error = position_MAE(
        torch.tensor(row['pred_cell_positions_prior']),
        torch.tensor(row['true_cell_positions']),
    ).numpy()
    prior_frac_positional_error_list.append(prior_frac_positional_error.item())
    
    positional_error = position_MAE(
        torch.tensor(pred_abs_positions),
        torch.tensor(gt_abs_positions),
    ).numpy()
    positional_error_list.append(positional_error.item())
    
    prior_positional_error = position_MAE(
        torch.tensor(prior_pred_abs_positions),
        torch.tensor(gt_abs_positions),
    ).numpy()
    prior_positional_error_list.append(prior_positional_error.item())
    
    # Calculate cell length and angle error
    cell_length_error, cell_angle_error = calculate_cell_errors(
        row['pred_cell_parameters'],
        cell_params,
    )
    cell_length_error_list.append(cell_length_error)
    cell_angle_error_list.append(cell_angle_error)
    
    prior_cell_length_error, prior_cell_angle_error = calculate_cell_errors(
        row['pred_cell_parameters_prior'],
        cell_params,
    )
    prior_cell_length_error_list.append(prior_cell_length_error)
    prior_cell_angle_error_list.append(prior_cell_angle_error)
    
    # Calculate atom accuracy
    atom_accuracy = calculate_atom_accuracy(row['pred_cell_atoms'], row['true_cell_atoms'])
    atom_accuracy_list.append(atom_accuracy)
    
    prior_atom_accuracy = calculate_atom_accuracy(row['pred_cell_atoms_prior'], row['true_cell_atoms'])
    prior_atom_accuracy_list.append(prior_atom_accuracy)
    
# Add positional error to dataframe
df_combined['frac_positional_error (no unit)'] = frac_positional_error_list
df_combined['positional_error (Å)'] = positional_error_list
df_combined['cell_length_error (Å)'] = cell_length_error_list
df_combined['cell_angle_error (°)'] = cell_angle_error_list
df_combined['atom_accuracy'] = atom_accuracy_list

df_combined['prior_frac_positional_error (no unit)'] = prior_frac_positional_error_list
df_combined['prior_positional_error (Å)'] = prior_positional_error_list
df_combined['prior_cell_length_error (Å)'] = prior_cell_length_error_list
df_combined['prior_cell_angle_error (°)'] = prior_cell_angle_error_list
df_combined['prior_atom_accuracy'] = prior_atom_accuracy_list

  0%|          | 0/318 [00:00<?, ?it/s]

In [18]:
df_combined[['crystalType', 'cell_parameters', 'cell_atoms', 'cell_positions', 'frac_positional_error (no unit)', 'positional_error (Å)', 'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy', 'prior_frac_positional_error (no unit)', 'prior_positional_error (Å)', 'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)', 'prior_atom_accuracy']].groupby('crystalType').mean()

,cell_parameters,cell_atoms,cell_positions,frac_positional_error (no unit),positional_error (Å),cell_length_error (Å),cell_angle_error (°),atom_accuracy,prior_frac_positional_error (no unit),prior_positional_error (Å),prior_cell_length_error (Å),prior_cell_angle_error (°),prior_atom_accuracy
crystalType,,,,,,,,,,,,,
AntiFluorite,0.018310,5.950958e-06,0.000375,0.029546,0.163594,0.262822,1.544381,100.0,0.060840,0.349198,0.237091,1.725093,100.000000
CadmiumChloride,0.006337,3.961793e-06,0.000711,0.040700,0.343457,0.187880,1.882325,100.0,0.038559,0.326897,0.224188,1.837916,100.000000
CadmiumIodide,0.005932,2.962803e-05,0.000926,0.031726,0.118395,0.177799,0.582625,100.0,0.084241,0.280787,0.169874,0.570346,98.351648
CaesiumChloride,0.004542,8.879402e-07,0.001993,0.062737,0.174471,0.170111,1.748534,100.0,0.119238,0.375023,0.285708,1.839283,98.148148
Fluorite,0.006437,3.220321e-06,0.000317,0.025262,0.136836,0.145869,1.224299,100.0,0.069066,0.403871,0.135684,1.267264,98.015873
NickelArsenide,0.005980,5.161307e-07,0.001044,0.045258,0.181902,0.143186,1.235152,100.0,0.059594,0.231836,0.121333,1.262902,100.000000
RheniumTrioxide,0.000394,3.654095e-06,0.000321,0.027873,0.108500,0.047617,1.112717,100.0,0.029896,0.116392,0.050363,1.099668,100.000000
RockSalt,0.004955,3.175865e-04,0.001811,0.046224,0.205482,0.150397,1.744412,100.0,0.033310,0.151771,0.176541,1.637972,100.000000
Rutile,0.002536,4.756887e-08,0.006772,0.054121,0.236424,0.130053,2.641473,100.0,0.053380,0.233290,0.131379,2.706767,100.000000


In [19]:
df_combined[['crystalType', 'cell_parameters', 'cell_atoms', 'cell_positions', 'frac_positional_error (no unit)', 'positional_error (Å)', 'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy', 'prior_frac_positional_error (no unit)', 'prior_positional_error (Å)', 'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)', 'prior_atom_accuracy']].groupby('crystalType').std()

,cell_parameters,cell_atoms,cell_positions,frac_positional_error (no unit),positional_error (Å),cell_length_error (Å),cell_angle_error (°),atom_accuracy,prior_frac_positional_error (no unit),prior_positional_error (Å),prior_cell_length_error (Å),prior_cell_angle_error (°),prior_atom_accuracy
crystalType,,,,,,,,,,,,,
AntiFluorite,0.029477,7.057250e-06,0.000151,0.005855,0.044669,0.253020,0.280522,0.0,0.093819,0.554728,0.228480,0.366341,0.000000
CadmiumChloride,0.009332,8.354876e-06,0.000335,0.009086,0.102566,0.163973,0.242270,0.0,0.006936,0.086467,0.185481,0.205023,0.000000
CadmiumIodide,0.008299,1.484736e-04,0.003354,0.034449,0.128275,0.141787,0.404324,0.0,0.306136,0.967419,0.140564,0.391892,8.404977
CaesiumChloride,0.006340,3.329003e-06,0.001105,0.015116,0.048053,0.104169,0.525946,0.0,0.305413,1.085077,0.590234,0.397894,9.622504
Fluorite,0.012550,1.124607e-05,0.000398,0.010720,0.064534,0.161073,0.372082,0.0,0.199090,1.211449,0.163146,0.472537,10.309826
NickelArsenide,0.008913,1.104557e-06,0.000885,0.016701,0.075693,0.109276,0.250057,0.0,0.084083,0.283897,0.080048,0.275590,0.000000
RheniumTrioxide,0.000615,7.121597e-06,0.000209,0.006694,0.026068,0.027173,0.152028,0.0,0.013187,0.051452,0.033044,0.203137,0.000000
RockSalt,0.007051,1.096095e-03,0.004361,0.047990,0.201790,0.113463,0.379950,0.0,0.011317,0.059120,0.134506,0.363288,0.000000
Rutile,0.001513,1.194179e-07,0.020565,0.033924,0.158977,0.042053,0.270216,0.0,0.034162,0.159851,0.037310,0.225930,0.000000


In [20]:
pd.options.display.float_format = '{:.3f}'.format
df_combined[['crystalType', 'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy', 'positional_error (Å)', 'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)', 'prior_atom_accuracy', 'prior_positional_error (Å)']].groupby('crystalType').median()

,cell_length_error (Å),cell_angle_error (°),atom_accuracy,positional_error (Å),prior_cell_length_error (Å),prior_cell_angle_error (°),prior_atom_accuracy,prior_positional_error (Å)
crystalType,,,,,,,,
AntiFluorite,0.177,1.546,100.000,0.154,0.145,1.631,100.000,0.155
CadmiumChloride,0.120,1.836,100.000,0.319,0.176,1.786,100.000,0.320
CadmiumIodide,0.126,0.434,100.000,0.085,0.099,0.444,100.000,0.082
CaesiumChloride,0.128,1.650,100.000,0.160,0.154,1.770,100.000,0.165
Fluorite,0.063,1.267,100.000,0.122,0.077,1.272,100.000,0.117
NickelArsenide,0.118,1.205,100.000,0.144,0.109,1.182,100.000,0.148
RheniumTrioxide,0.046,1.105,100.000,0.108,0.047,1.048,100.000,0.110
RockSalt,0.126,1.724,100.000,0.138,0.143,1.716,100.000,0.142
Rutile,0.131,2.636,100.000,0.201,0.132,2.701,100.000,0.193


In [21]:
df_combined[['cell_parameters', 'cell_atoms', 'cell_positions', 'kld', 'frac_positional_error (no unit)', 'positional_error (Å)', 'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy', 'prior_frac_positional_error (no unit)', 'prior_positional_error (Å)', 'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)', 'prior_atom_accuracy']].describe().loc[['mean', 'std']].T

,mean,std
cell_parameters,0.006,0.013
cell_atoms,0.000,0.000
cell_positions,0.002,0.007
kld,0.093,1.072
frac_positional_error (no unit),0.041,0.028
positional_error (Å),0.202,0.144
cell_length_error (Å),0.165,0.149
cell_angle_error (°),1.668,0.661
atom_accuracy,100.000,0.000
prior_frac_positional_error (no unit),0.060,0.163


In [22]:
pd.options.display.float_format = '{:.7f}'.format
df_combined[['cell_parameters', 'cell_atoms', 'cell_positions', 'kld', 'frac_positional_error (no unit)', 'positional_error (Å)', 'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy', 'prior_frac_positional_error (no unit)', 'prior_positional_error (Å)', 'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)', 'prior_atom_accuracy']].describe(percentiles=[0.25,0.5, 0.75]).T

,count,mean,std,min,25%,50%,75%,max
cell_parameters,318.0000000,0.0064997,0.0127785,0.0000280,0.0005798,0.0019529,0.0064658,0.1083654
cell_atoms,318.0000000,0.0000346,0.0003291,0.0000000,0.0000000,0.0000010,0.0000036,0.0053856
cell_positions,318.0000000,0.0015263,0.0065711,0.0000899,0.0002839,0.0004611,0.0009452,0.0801378
kld,318.0000000,0.0929320,1.0722216,0.0000014,0.0000934,0.0003437,0.0010764,17.6910744
frac_positional_error (no unit),318.0000000,0.0410649,0.0279349,0.0151318,0.0263727,0.0324798,0.0452417,0.2188134
positional_error (Å),318.0000000,0.2017620,0.1437915,0.0658753,0.1188233,0.1483161,0.2390811,0.9143176
cell_length_error (Å),318.0000000,0.1645778,0.1488220,0.0097383,0.0609862,0.1199510,0.2075813,0.8962499
cell_angle_error (°),318.0000000,1.6677139,0.6614006,0.1632524,1.2094437,1.6888250,2.0765054,3.2737440
atom_accuracy,318.0000000,100.0000000,0.0000000,100.0000000,100.0000000,100.0000000,100.0000000,100.0000000
prior_frac_positional_error (no unit),318.0000000,0.0597614,0.1629839,0.0156481,0.0266793,0.0326736,0.0441333,1.6469629


In [23]:
for col, median in df_combined[['cell_parameters', 'cell_atoms', 'cell_positions', 'kld', 'frac_positional_error (no unit)', 'positional_error (Å)', 'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy', 'prior_frac_positional_error (no unit)', 'prior_positional_error (Å)', 'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)', 'prior_atom_accuracy']].median().items():
    print(f"{col:<50} {median:>10.5f}")
    print('')

cell_parameters                                       0.00195

cell_atoms                                            0.00000

cell_positions                                        0.00046

kld                                                   0.00034

frac_positional_error (no unit)                       0.03248

positional_error (Å)                                  0.14832

cell_length_error (Å)                                 0.11995

cell_angle_error (°)                                  1.68882

atom_accuracy                                       100.00000

prior_frac_positional_error (no unit)                 0.03267

prior_positional_error (Å)                            0.15040

prior_cell_length_error (Å)                           0.12889

prior_cell_angle_error (°)                            1.70387

prior_atom_accuracy                                 100.00000



## Plot

In [24]:
# Set seaborn style
# sns.set_theme(style='darkgrid', font_scale=1.)
# Make animation of latent space throughout 
# Plot latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_means.pdf', dpi=300)

In [25]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_combined[contour_column].min()
max_loss = df_combined[contour_column].max()

# Plot
if len(df_combined['latent_space_mean'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['ls1'].min()*1.1, df_combined['ls1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['ls2'].min()*1.1, df_combined['ls2'].max()*1.1, 1000)
    zi = griddata((df_combined['ls1'], df_combined['ls2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['pc1'].min()*1.1, df_combined['pc1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['pc2'].min()*1.1, df_combined['pc2'].max()*1.1, 1000)
    zi = griddata((df_combined['pc1'], df_combined['pc2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()


In [26]:
# # 2D scatter plot with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         pass
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace)
        
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types):
#             fig.data[i]['visible'] = True
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j, crystal_type in enumerate(crystal_types):
#                 step['args'][1][i*n_crystal_types + j] = True
#             steps.append(step)    
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.show()

In [27]:
# Sample n points from each distribution in the latent space log
if df_latent_log is not None:
    n_samples = 100
    
    crystal_type_list = []
    latent_space_list = []
    epoch_list = []
    for latent_mean, latent_std, crystal_type, epoch in zip(df_latent_log['posterior_mean'], df_latent_log['posterior_std'], df_latent_log['crystal_type'], df_latent_log['epoch']):
        latent_space_samples = torch.distributions.Normal(loc=torch.tensor(latent_mean), scale=torch.tensor(latent_std)).sample((n_samples,)).numpy()
        latent_space_list.extend(latent_space_samples)
        crystal_type_list.extend([crystal_type] * n_samples)
        epoch_list.extend([epoch] * n_samples)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_log_samples = pd.DataFrame(latent_space_list, columns=['ls1', 'ls2'])
    else:
        latent_space_list = pca.transform(latent_space_list)
        df_log_samples = pd.DataFrame(latent_space_list, columns=['pc1', 'pc2'])

    df_log_samples['crystalType'] = crystal_type_list
    df_log_samples['epoch'] = epoch_list

In [28]:
# # 2D scatter plot and 2D histogram with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['ls1'],
#                     y=df_crystal['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['ls1'],
#                     y=df_crystal_samples['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
            
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)
            
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='LS dim 1')
#         fig.update_yaxes(title_text='LS dim 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['pc1'],
#                     y=df_crystal_samples['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
        
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)  
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
    

In [29]:
# Plot the latent space at specific epochs
epoch_to_plot = 0

if df_latent_log is not None:
    plt.figure(figsize=(10, 8))
    df_epoch = df_latent_log[df_latent_log['epoch'] == epoch_to_plot]
    df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch_to_plot]
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        sns.scatterplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['ls1'].min(), df_log_samples['ls1'].max())
        plt.ylim(df_log_samples['ls2'].min(), df_log_samples['ls2'].max())  
    else:
        sns.scatterplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['pc1'].min(), df_log_samples['pc1'].max())
        plt.ylim(df_log_samples['pc2'].min(), df_log_samples['pc2'].max())
    
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    # plt.axis('equal')
    plt.tight_layout()
    plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epoch_{epoch_to_plot}.pdf', dpi=300)

In [30]:
# Make a combined figure of the latent space at different epochs
epochs_to_plot = [0, 10, 100, 350, 690, 890]
subplot_rows = 2
subplot_cols = 3
figsize = (10, 6.5)

if df_latent_log is not None:
    fig, ax = plt.subplots(subplot_rows, subplot_cols, figsize=figsize, sharex=True, sharey=True)
    ax = ax.flatten()
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        xlim_min = df_log_samples['ls1'].min()
        xlim_max = df_log_samples['ls1'].max()
        ylim_min = df_log_samples['ls2'].min()
        ylim_max = df_log_samples['ls2'].max()
    else:
        xlim_min = df_log_samples['pc1'].min()
        xlim_max = df_log_samples['pc1'].max()
        ylim_min = df_log_samples['pc2'].min()
        ylim_max = df_log_samples['pc2'].max()
    
    for i, epoch in enumerate(epochs_to_plot):
        df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
        df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
        
        if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
            sns.histplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        else:
            sns.histplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('PC 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('PC 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        if i == 0:
            handles, labels = ax[i].get_legend_handles_labels()
            ax[i].legend(loc='lower center', bbox_to_anchor=(subplot_cols/2, 1), ncol=4)
        else:
            ax[i].legend([],[], frameon=False)
        ax[i].set_title(f' Epoch {epoch}', fontsize=12, fontweight='bold', y=1.0, pad=-15, loc='left')
    
    # Remove empty axes
    for i in range(len(epochs_to_plot), subplot_rows*subplot_cols):
        fig.delaxes(ax[i])
        
    # fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 3), ncols=5)
    
    plt.tight_layout()
    
    # Remove whitespace
    plt.subplots_adjust(wspace=0.02, hspace=0.02)
    
    plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epochs.pdf', dpi=300)

# Experimental data

## Code

In [31]:
data_folder = './data/Experimental/'

# Load experimental data
data_paths = [str(p) for p in pathlib.Path(data_folder).rglob('*.gr')]

data_filepath = []
data_composition_string = []
data_composition = []
data_pdf = []
for data_path in data_paths:
    with open(data_path, 'r') as f:
        # Load data
        line_counter = 0
        for line in f:
            if line.startswith('composition'):
                composition = ''.join(line.split(' ')[2:])
            if line.startswith('0'):
                header_line = line_counter
                break
            line_counter += 1
    # Remove line breaks
    composition = composition.replace('\n', '')
    
    # Save composition string
    composition_string = deepcopy(composition)

    # Remove stochiometry from composition
    composition = re.sub(r'[0-9\.]+', '', composition)

    # # Split string on capital letters
    composition = re.findall('[A-Z][^A-Z]*', composition)

    # Translate composition to atom numbers
    composition = Atoms(symbols=composition).get_atomic_numbers()
    
    composition_onehot = np.zeros(119)
    composition_onehot[composition] = 1
    
    
    # Load data
    data = pd.read_csv(data_path, sep=' ', skiprows=header_line, names=['r [Å]', 'G(r) [Å⁻²]'])
    
    data_r = np.arange(0,60,0.01)
    data_Gr = np.interp(data_r, data['r [Å]'], data['G(r) [Å⁻²]'], left=0, right=0)
    data_Gr = data_Gr / np.amax(data_Gr)
    
    data_filepath.append(data_path)
    data_composition_string.append(composition_string)
    data_composition.append(composition_onehot)
    data_pdf.append(data_Gr)

# Convert to tensors
data_composition = torch.tensor(data_composition, dtype=torch.long)
data_pdf = torch.tensor(data_pdf, dtype=torch.float32)
data_composition_string_index = torch.tensor(np.arange(len(data_composition_string)))
data_filepath_index = torch.tensor(np.arange(len(data_filepath)))

exp_data = TensorDataset(data_pdf, data_composition, data_composition_string_index, data_filepath_index)

# Dataloader
exp_loader = DataLoader(exp_data, batch_size=10, shuffle=False)

# Results dict
exp_results = {
    'composition': [],
    'pdf': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [32]:
# Inference
model.eval()
for batch in tqdm(exp_loader, desc='Inference', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    pdf, composition, composition_string_index, filepath_index = batch
    pdf = pdf.unsqueeze(-1).to(device)
    composition = composition.float().to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store composition
    for index in composition_string_index:
        exp_results['composition'].append(data_composition_string[index])
    
    # Store PDF
    exp_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store latent representation
    exp_results['prior_mean'].extend(prior_mean.cpu().tolist())
    exp_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    exp_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    exp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    exp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    exp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
                prediction=True,
                composition=data_composition_string[composition_string_index[batch_index]],
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_exp_results = pd.DataFrame(exp_results)
if len(df_exp_results['prior_mean'].iloc[0]) == 2:
    df_exp_results[['ls1', 'ls2']] = df_exp_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_exp_results[['pc1', 'pc2']] = pca.transform(np.array(df_exp_results['prior_mean'].values.tolist()))
df_exp_results.head()

# Drop IrO2
# df_exp_results = df_exp_results[df_exp_results['composition'] != 'IrO2']

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[0.34958064556121826, -1.4048129320144653]","[0.12149949371814728, 0.17361588776111603]","[0.0590512752532959, -1.3987634181976318]","[4.596202373504639, 4.632835388183594, 2.87167...","[[0.04129000008106232, -0.03585999831557274, 0...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.3495806,-1.4048129
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[-0.8993464112281799, 0.10989199578762054]","[0.07034019380807877, 0.14894959330558777]","[-0.8272374272346497, 0.10075908899307251]","[8.476621627807617, 8.472660064697266, 8.42813...","[[0.5142099857330322, 0.5121899843215942, 0.50...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.8993464,0.1098920
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[0.35640615224838257, -1.437368631362915]","[0.11025744676589966, 0.16118021309375763]","[0.2925814092159271, -1.1656876802444458]","[4.631579399108887, 4.635101795196533, 2.95615...","[[0.012000000104308128, -0.0036100000143051147...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.3564062,-1.4373686
3,Cr0.66Mn0.66Fe0.66Ni0.5Cu0.5O4,"[[0.0], [3.48158209817484e-05], [6.78441429045...","[-0.4976503252983093, -0.5985623002052307]","[0.08616134524345398, 0.14953884482383728]","[-0.4540221095085144, -0.5289234519004822]","[5.586108207702637, 5.600020885467529, 5.57305...","[[-0.0014900000533089042, 0.005940000060945749...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.4976503,-0.5985623
4,Co0.33Ni0.33Zn0.33CrMnO4,"[[0.0], [2.2900278509041527e-06], [4.542134774...","[-0.6564680933952332, -0.18515054881572723]","[0.06771566718816757, 0.13834622502326965]","[-0.6105645895004272, -0.07228562980890274]","[7.366678237915039, 7.280555248260498, 7.15318...","[[0.24844999611377716, 0.25964999198913574, 0....","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.6564681,-0.1851505


In [33]:
df_exp_results['ground_truth_crystal_type'] = None
df_exp_results['ground_truth_crystal_type'].iloc[0:3] = 'Rutile'
df_exp_results['ground_truth_crystal_type'].iloc[3:6] = 'Fluorite'
df_exp_results['ground_truth_crystal_type'].iloc[6:11] = 'Spinel'

## Plot

In [34]:
# Plot latent space placement of experimental data
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='ls1', y='ls2', hue='composition', style='ground_truth_crystal_type', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='pc1', y='pc2', c='black', style='composition', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_samples.pdf', dpi=300)

In [35]:
# Plot the experimental data
index_list = [0,1,2] # Rutile samples
# index_list = [3, 4, 5] # Fluorite samples
# index_list = list(range(6,11)) # Spinel samples

plt.figure(figsize=(8, 6))
for index in index_list:
    plt.plot(np.arange(0,60,0.01), np.array(df_exp_results['pdf'].iloc[index]) + (index * 1.25), label=df_exp_results['composition'].iloc[index])
plt.xlabel('r [Å]', fontsize=14, fontweight='bold')
plt.ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
plt.yticks([])
# plt.legend(ncols=2, loc='lower center', bbox_to_anchor=(0.5, 1.01))
plt.legend(ncols=3, loc='lower center', bbox_to_anchor=(0.5, 1.01))
# plt.title(f'{df_exp_results["composition"].iloc[index]}', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Rutile.pdf', dpi=300)
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Fluorite.pdf', dpi=300)
# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Spinel.pdf', dpi=300)

In [36]:
# Plot experimental samples in 3 side-by-side subplots. The rightmost subplot should be twice as wide as the others.

# Make plotgrid with 1 row and 3 columns
fig, ax = plt.subplots(1, 3, figsize=(10, 5), gridspec_kw={'width_ratios': [1, 1, 2]})

# Plot Rutile samples
for index in [0, 1, 2]:
    ax[0].plot(np.arange(0,60,0.01), np.array(df_exp_results['pdf'].iloc[index]) + (index * 1.25), label=df_exp_results['composition'].iloc[index])
    
ax[0].set_xlabel('r [Å]', fontsize=14, fontweight='bold')
ax[0].set_ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
ax[0].set_xlim(0, 30)
ax[0].set_title('IrO$_2$ (Rutile)', fontsize=16, fontweight='bold')
ax[0].annotate('a)', xy=(0.01, 0.98), xycoords='axes fraction', fontsize=16, fontweight='bold', ha='left', va='top')

# Plot Fluorite samples
for index in [3, 4, 5]:
    ax[1].plot(np.arange(0,60,0.01), np.array(df_exp_results['pdf'].iloc[index]) + (index * 1.25), label=df_exp_results['composition'].iloc[index])
ax[1].set_xlabel('r [Å]', fontsize=14, fontweight='bold')
ax[1].set_xlim(0, 30)
ax[1].set_title('CeO$_2$ (Fluorite)', fontsize=16, fontweight='bold')
ax[1].annotate('b)', xy=(0.01, 0.98), xycoords='axes fraction', fontsize=16, fontweight='bold', ha='left', va='top')

# Plot Spinel samples
for index in [6, 7, 8, 9, 10]:
    ax[2].plot(np.arange(0,60,0.01), np.array(df_exp_results['pdf'].iloc[index]) + (index * 1.25), label=df_exp_results['composition'].iloc[index])
ax[2].set_xlabel('r [Å]', fontsize=14, fontweight='bold')
ax[2].set_title('HEO (Spinel)', fontsize=16, fontweight='bold')
ax[2].annotate('c)', xy=(0.01, 0.98), xycoords='axes fraction', fontsize=16, fontweight='bold', ha='left', va='top')

ax[2].set_xlim(0, 60)

# Set y-ticks to empty
ax[0].set_yticks([])
ax[1].set_yticks([])
ax[2].set_yticks([])

# Set legend
# ax[0].legend(ncols=3, loc='lower center', bbox_to_anchor=(0.5, 1.01))
# ax[1].legend(ncols=3, loc='lower center', bbox_to_anchor=(0.5, 1.01))
# ax[2].legend(ncols=2, loc='lower center', bbox_to_anchor=(0.5, 1.01))

# Remove whitespace between subplots

plt.tight_layout()
plt.subplots_adjust(wspace=0.013, hspace=0.0)

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_collective.pdf', dpi=300)


# Interpolate in latent space

## Code

In [38]:
# NiAs -> CdI2 -> CdCl2
# n_latent_samples = 7
# crystal_types = ['NickelArsenide', 'CadmiumIodide', 'CadmiumChloride'] #['CadmiumIodide', 'CadmiumChloride'] #['RockSalt', 'Spinel', 'ZincBlende']  ['NickelArsenide', 'CadmiumIodide', 'CadmiumChloride']
# indices = [0, 4, 3] #[14,3] #[1, 4, 0]  [0, 4, 3]

# RockSalt -> Spinel -> ZincBlende
# n_latent_samples = 7
# crystal_types = ['RockSalt', 'Spinel', 'ZincBlende']
# indices = [3, 2, 2]

# RheniumTrioxide -> RockSalt
n_latent_samples = 7
crystal_types = ['RheniumTrioxide','RockSalt']
indices = [0, 4]

point_indices = [df_rec[(df_rec['crystalType'] == crystal_type)].index[index] for crystal_type, index in zip(crystal_types, indices)]

latent_points = [np.array(df_rec['latent_space_mean'].iloc[point_index]) for point_index in point_indices]

# Interpolation mode flag (True for PCA. False for full latent space)
pca_inter = False

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Latent samples are n_latent_samples evenly spaced points between the two points
if pca_inter:
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_pca = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_pca = np.concatenate([interp_pca, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples)[1:] for j in range(len(latent_points[i]))]).T], axis=0)
    
    if len(df_rec['latent_space_mean'].iloc[0]) > 2:
        interp_latent = pca.inverse_transform(interp_pca)
    else:
        interp_latent = interp_pca
    interp_latent = torch.tensor(interp_latent).float()
else:
    # Transform back to latent dimensions if latent space is more than 2D
    if (len(df_rec['latent_space_mean'].iloc[0]) > 2) and (len(latent_points[0]) == 2):
        latent_points = pca.inverse_transform(latent_points)
    
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_latent = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_latent = np.concatenate([interp_latent, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples)[1:] for j in range(len(latent_points[i]))]).T], axis=0)
        
    interp_latent = torch.tensor(interp_latent).float()

n_interp_points = interp_latent.shape[0]
interp_index = torch.tensor([i for i in range(n_interp_points)])

# Create dataset
interp_dataset = TensorDataset(interp_latent, interp_index, comp_onehot.repeat(n_interp_points, 1))

# Dataloader
interp_loader = DataLoader(interp_dataset, batch_size=10, shuffle=False)

# Results dict
interp_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [39]:
# Inference
model.eval()
for batch in tqdm(interp_loader, desc='Interpolating', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    interp_point, interp_index, composition = batch
    interp_point = interp_point.to(device)
    interp_index = interp_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            interp_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    interp_results['name'].extend([f'sample_{interp_index[batch_index]}' for batch_index in range(this_batch_size)])
    interp_results['latent_point'].extend(interp_point.cpu().tolist())
    
    # Store predictions
    interp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions/sample{interp_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {interp_index[batch_index]}.')
    

In [40]:
df_interp = pd.DataFrame(interp_results)
if len(df_interp['latent_point'].iloc[0]) == 2:
    df_interp[['ls1', 'ls2']] = df_interp['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_interp[['pc1', 'pc2']] = pca.transform(np.array(df_interp['latent_point'].values.tolist()))
df_interp['Interpolation number'] = df_interp['name'].apply(lambda x: int(x.split('_')[1]))
df_interp.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2,Interpolation number
0,sample_0,"[-0.32636791467666626, 0.9011446833610535]","[3.875000238418579, 3.877912998199463, 3.79130...","[[0.0008399999933317304, 0.004689999856054783,...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",-0.3263679,0.9011447,0
1,sample_1,"[-0.3016313910484314, 0.7920211553573608]","[3.8796586990356445, 3.8769097328186035, 3.849...","[[0.0002500000118743628, 0.004989999812096357,...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",-0.3016314,0.7920212,1
2,sample_2,"[-0.27689486742019653, 0.6828975677490234]","[3.9326252937316895, 3.908158779144287, 3.9284...","[[-0.005319999996572733, -0.008580000139772892...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",-0.2768949,0.6828976,2
3,sample_3,"[-0.25215834379196167, 0.5737740397453308]","[4.022838115692139, 4.00612211227417, 3.986935...","[[0.005880000069737434, -0.02734999917447567, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",-0.2521583,0.5737740,3
4,sample_4,"[-0.2274218201637268, 0.4646504819393158]","[4.159781455993652, 4.193562984466553, 4.05162...","[[0.012679999694228172, -0.016759999096393585,...","[2, 1, 1, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, ...",-0.2274218,0.4646505,4


## Plot

In [41]:
# Plot interpolation results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order)
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order)
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_samples.pdf', dpi=300)

In [42]:
# Plot the cell parameters of the interpolation with angles and lengths on two subplots (one for lengths and one for angles) that share the x-axis
df_cell_params = pd.DataFrame(df_interp['cell_parameters'].tolist(), columns=['a', 'b', 'c', 'alpha', 'beta', 'gamma'])
df_cell_params['name'] = df_interp['name']

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

# Lengths
sns.lineplot(data=df_cell_params[['a', 'b', 'c']], ax=ax0, zorder=10, lw=2)
ax0.set_ylabel('Length [Å]', fontsize=14, fontweight='bold')

# Angles
sns.lineplot(data=df_cell_params[['alpha', 'beta', 'gamma']], ax=ax1, zorder=10, lw=2)
ax1.set_ylabel('Angle [°]', fontsize=14, fontweight='bold')
ax1.set_xlabel('Interpolation sample #', fontsize=14, fontweight='bold')
# y-axis label and ticks on the right side
ax1.yaxis.tick_right()
ax1.yaxis.set_label_position("right")

# Add vertical lines for the interpolation outside clusters and color between them
# NiAs -> CdI2 -> CdCl2
# line_1 = 3.5 
# line_2 = 4.5 
# line_3 = 8.5 
# line_4 = 10.5

# RockSalt -> Spinel -> ZincBlende
# line_1 = 1.5 
# line_2 = 4.5 
# line_3 = 8.5 
# line_4 = 10.5

# RheniumTrioxide -> RockSalt
line_1 = 2.5 
line_2 = 4.5 
line_3 = 8.5 
line_4 = 10.5

ax0.axvline(x=line_1, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvline(x=line_2, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvspan(line_1, line_2, color='red', alpha=0.1, zorder=1)

ax0.axvline(x=line_3, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvline(x=line_4, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvspan(line_3, line_4, color='red', alpha=0.1, zorder=1)

ax1.axvline(x=line_1, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvline(x=line_2, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvspan(line_1, line_2, color='red', alpha=0.1, zorder=1)

ax1.axvline(x=line_3, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvline(x=line_4, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvspan(line_3, line_4, color='red', alpha=0.1, zorder=1)

plt.xlim(0, n_interp_points-1)

# Legends
ax0.legend(loc='center left' , bbox_to_anchor=(0.01, 0.5), ncol=1)
ax1.legend(loc='center left' , bbox_to_anchor=(0.01, 0.5), ncol=1)

# Make x-ticks integers
plt.xticks(np.arange(0, n_interp_points, 1))

ax1.set_ylim((85,125))

fig.tight_layout()

# Remove space between subplots
plt.subplots_adjust(hspace=0)

plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/interpolation_cell_parameters.pdf', dpi=300)

# Sample same distribution multiple times

## Code

In [43]:
# Number of samples to generate
n_samples = 10

# Latent distribution to sample from
dist_crystal_type = 'CadmiumIodide'
crystal_type_index = 0
dist_index = df_rec[(df_rec['crystalType'] == dist_crystal_type)].index[crystal_type_index]
dist_mean = df_rec.latent_space_mean_prior.iloc[dist_index]
dist_std = df_rec.latent_space_std_prior.iloc[dist_index]

# Latent distribution from experimental data
# exp_index = 3
# dist_mean = df_exp_results.prior_mean.iloc[exp_index]
# dist_std = df_exp_results.prior_log_std.iloc[exp_index]

latent_dist = torch.distributions.Normal(loc=torch.tensor(dist_mean).float(), scale=torch.tensor(dist_std).float())

# Sample from latent distribution
latent_mean = latent_dist.mean
latent_samples = latent_dist.sample((n_samples,))
# Concatenate with latent mean
latent_samples = torch.cat((latent_mean.unsqueeze(0), latent_samples), dim=0)
sample_index = torch.tensor([i for i in range(n_samples+1)])

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Define dataset
sample_dataset = TensorDataset(latent_samples, sample_index, comp_onehot.repeat(n_samples+1, 1))

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [44]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition = batch
    sample_point = sample_point.to(device)
    sample_index = sample_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    sample_results['name'].extend([f'sample_{sample_index[batch_index]}' for batch_index in range(this_batch_size)])
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions/sample{sample_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {sample_index[batch_index]}.')

In [45]:
df_sample = pd.DataFrame(sample_results)
if len(df_sample['latent_point'].iloc[0]) == 2:
    df_sample[['ls1', 'ls2']] = df_sample['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_sample[['pc1', 'pc2']] = pca.transform(np.array(df_sample['latent_point'].values.tolist()))
    
df_sample['name'][df_sample['name'] == 'sample_0'] = 'Dist. mean'
df_sample.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,Dist. mean,"[0.6561934947967529, 1.5598856210708618]","[3.059079647064209, 3.0595059394836426, 4.3979...","[[-0.0026100000832229853, 0.001150000025518238...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.6561935,1.5598856
1,sample_1,"[0.629041314125061, 1.5578192472457886]","[3.057964324951172, 3.060706853866577, 4.37652...","[[-0.0022899999748915434, 0.001260000048205256...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.6290413,1.5578192
2,sample_2,"[0.7471446394920349, 1.5552589893341064]","[3.0680325031280518, 3.0618906021118164, 4.461...","[[-0.002570000011473894, 0.0005699999746866524...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.7471446,1.5552590
3,sample_3,"[0.6804209351539612, 1.5519698858261108]","[3.0537285804748535, 3.0525479316711426, 4.402...","[[-0.0026499999221414328, 0.000679999997373670...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.6804209,1.5519699
4,sample_4,"[0.6183009743690491, 1.803524136543274]","[3.3827946186065674, 3.4097282886505127, 5.028...","[[-0.005239999853074551, 0.009580000303685665,...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.6183010,1.8035241


## Plot

In [46]:
# Plot sampling results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='ls1', y='ls2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='ls1', y='ls2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('LS dim 1')
    plt.ylabel('LS dim 2')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    # sns.scatterplot(data=df_sample, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='pc1', y='pc2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='pc1', y='pc2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1')
    plt.ylabel('PC 2')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sampling_samples.pdf', dpi=300)

# Sample every latent distrubution and calculate loss

In [47]:
# Number of samples per point
n_samples = 100 #1000
# test_data = 'validation'

# Seed for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# Sample from each latent distribution
data_names = []
data_samples = []
data_sample_type = []
data_crystal_types = []
ground_truth_cell_parameters = []
ground_truth_cell_positions = []
ground_truth_cell_atoms = []
for data_index in range(len(df_rec)):
    latent_mean = df_rec['latent_space_mean'].iloc[data_index]
    latent_std = df_rec['latent_space_std'].iloc[data_index]
    latent_dist = torch.distributions.Normal(loc=torch.tensor(latent_mean).float(), scale=torch.tensor(latent_std).float())
    latent_samples = latent_dist.sample((n_samples,))
    
    data_samples.append(torch.tensor(latent_mean))
    data_samples.extend([latent_sample for latent_sample in latent_samples])
    
    data_names.append(f'{data_index}')
    data_names.extend([f'{data_index}_{i}' for i in range(n_samples)])
    
    data_sample_type.append('mean')
    data_sample_type.extend(['sample' for i in range(n_samples)])
    
    data_crystal_types.extend([df_rec['crystalType'].iloc[data_index] for i in range(n_samples+1)])
    
    cell_positions_padded = np.zeros((setup_json['model']['out_dim'], 3))
    cell_positions_padded[:len(df_rec['pred_cell_positions'].iloc[data_index]), :] = np.array(df_rec['pred_cell_positions'].iloc[data_index])
    # if setup_json['training']['simplified_atom_identities']:
    #     cell_atoms_padded = torch.zeros((setup_json['model']['out_dim'], 3))
    # else:
    #     cell_atoms_padded = torch.zeros((setup_json['model']['out_dim'], 119))
    cell_atoms_padded = np.zeros((setup_json['model']['out_dim'],))
    cell_atoms_padded[:len(df_rec['pred_cell_atoms'].iloc[data_index])] = np.array(df_rec['pred_cell_atoms'].iloc[data_index])
    
    # Ground truth
    ground_truth_cell_parameters.extend([df_rec['pred_cell_parameters'].iloc[data_index]]*(n_samples+1))
    ground_truth_cell_positions.extend([cell_positions_padded]*(n_samples+1))
    ground_truth_cell_atoms.extend([cell_atoms_padded]*(n_samples+1))
    
data_samples = torch.stack(data_samples)
ground_truth_cell_parameters = torch.tensor(ground_truth_cell_parameters, dtype=torch.float32)
ground_truth_cell_positions = torch.tensor(ground_truth_cell_positions, dtype=torch.float32)
ground_truth_cell_atoms = torch.tensor(ground_truth_cell_atoms, dtype=torch.long)

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

data_index = torch.tensor(np.arange(len(data_names)))

# Define dataset
sample_dataset = TensorDataset(data_samples, data_index, comp_onehot.repeat(len(data_names), 1), ground_truth_cell_parameters, ground_truth_cell_positions, ground_truth_cell_atoms)

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'sample_type': [],
    'crystalType': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
    'reconstruction_loss': [],
}
    

In [48]:
# Loss functions
loss_fn_cell_parameters = torch.nn.MSELoss()
# loss_fn_cell_positions = torch.nn.MSELoss()
# loss_fn_cell_atoms = torch.nn.CrossEntropyLoss()
# loss_fn_kld = torch.nn.KLDivLoss()
loss_fn_cell_positions = weighted_MSELoss()
loss_fn_cell_atoms = weighted_CrossEntropyLoss()
loss_fn_latent_mean = torch.nn.MSELoss()
loss_fn_latent_std = torch.nn.MSELoss()

In [49]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition, cell_parameters_true, cell_positions_true, cell_atoms_true = batch
    sample_point = sample_point.to(device)
    sample_indeces = sample_index.to(device)
    composition = composition.to(device)
    cell_parameters_true = cell_parameters_true.to(device)
    cell_positions_true = cell_positions_true.to(device)
    cell_atoms_true = cell_atoms_true.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    for batch_index, sample_index in enumerate(sample_indeces):
        sample_results['name'].append(data_names[sample_index])
        sample_results['sample_type'].append(data_sample_type[sample_index])
        if data_crystal_types[sample_index] == 'Interpolated':
            sample_results['crystalType'].append(data_crystal_types[sample_index])#+ ' ' + str(sum(cell_atoms_true[batch_index] > 0).item()))
        else:
            sample_results['crystalType'].append(data_crystal_types[sample_index])
    
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Make loss weights
    cell_positions_weights = torch.where(cell_positions_true != -1, 1, 0).float().to(device)
    cell_atoms_weights = torch.where(cell_atoms_true != 0, 1, 0.1).float().to(device)

    # Simplify atom identities
    if setup_json['training']['simplified_atom_identities']:
        # Map atom number 0 to logit 0 (No atom)
        cell_atoms_true = torch.where(cell_atoms_true == 0, 0, cell_atoms_true)
        # Map atom numbers of ligands to logit 1 (Ligand) # [1, 6, 7, 8, 9, 15, 16, 17, 34, 35, 53]
        for ligand in setup_json['training']['ligands']:
            cell_atoms_true = torch.where(cell_atoms_true == ligand, 1, cell_atoms_true)
        # Map all other atom numbers to logit 2 (Metal)
        cell_atoms_true = torch.where(cell_atoms_true >= 2, 2, cell_atoms_true)
    
    # Calculate reconstruction loss for each sample
    for batch_index in range(this_batch_size):
        loss_cell_parameters = loss_fn_cell_parameters(cell_parameters[batch_index], cell_parameters_true[batch_index])
        loss_cell_positions = loss_fn_cell_positions(cell_positions[batch_index], cell_positions_true[batch_index], cell_positions_weights[batch_index])
        loss_cell_atoms = loss_fn_cell_atoms(cell_atoms[batch_index], cell_atoms_true[batch_index], cell_atoms_weights[batch_index])
        
        loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
        
        # Save loss
        sample_results['reconstruction_loss'].append(loss_reconstruction.item())
        
    # # Reconstruction loss
    # loss_cell_parameters = loss_fn_cell_parameters(cell_parameters, cell_parameters_true) 
    
    # # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true) # Unweighted
    # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true, cell_positions_weights) # Weighted
    
    # # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true) # Unweighted
    # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true, cell_atoms_weights) # Weighted
    
    # loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
    
    # # Save loss
    # sample_results['reconstruction_loss'].extend([loss_reconstruction.item()]*this_batch_size)


In [50]:
df_full_latent = pd.DataFrame(sample_results)
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    df_full_latent[['ls1', 'ls2']] = df_full_latent['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_full_latent[['pc1', 'pc2']] = pca.transform(np.array(df_full_latent['latent_point'].values.tolist())
)
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Rutile,"[0.34804725646972656, -1.4523036479949951]","[4.567151069641113, 4.565000534057617, 2.87790...","[[0.002739999908953905, -0.01370999962091446, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.0953129,0.3480473,-1.4523036
1,0_0,sample,Rutile,"[0.5179596543312073, -1.2605674266815186]","[4.494833469390869, 4.47055721282959, 2.888001...","[[0.001769999973475933, -0.007530000060796738,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.9170761,0.5179597,-1.2605674
2,0_1,sample,Rutile,"[0.4274711012840271, -1.723741054534912]","[4.5453596115112305, 4.533892631530762, 2.8714...","[[0.011939999647438526, -0.016189999878406525,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.3335764,0.4274711,-1.7237411
3,0_2,sample,Rutile,"[0.4078691303730011, -1.6114574670791626]","[4.5486297607421875, 4.542178153991699, 2.8620...","[[0.011769999749958515, -0.021460000425577164,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.1781759,0.4078691,-1.6114575
4,0_3,sample,Rutile,"[0.3442496359348297, -1.6591724157333374]","[4.554953098297119, 4.5517258644104, 2.8566992...","[[0.003389999968931079, -0.019300000742077827,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.2322302,0.3442496,-1.6591724


In [51]:
# Plot sampling results in latent space
if 'Interpolation' in model_path:
    custom_palette = ['k', sns.color_palette('tab20')[4], sns.color_palette('tab20')[7]]
    custom_markers = ['o', (4,0,0), (4,1,45)]
    if len(df_rec['latent_space_mean'].iloc[0]) == 2:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)
        
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette=custom_palette, markers=custom_markers)
        # sns.scatterplot(data=df_full_latent[(df_full_latent['sample_type'] == 'mean') & (df_full_latent['crystalType'].isin(['NickelArsenide', 'CadmiumIodide']))], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette=custom_palette[1:], markers=custom_markers[1:])
        # sns.scatterplot(data=df_full_latent[(df_full_latent['sample_type'] == 'mean') & ~(df_full_latent['crystalType'].isin(['NickelArsenide', 'CadmiumIodide']))], x='ls1', y='ls2', hue='crystalType', marker='o', s=50, palette='viridis')
        
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    else:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette=custom_palette, markers=custom_markers)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
else:
    if len(df_rec['latent_space_mean'].iloc[0]) == 2:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)    
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order, style_order=legend_order)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    else:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order, style_order=legend_order)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.pdf', dpi=300)

# Save svg for reconstruction figure

# Remove axis border, labels and ticks
plt.axis('off')
# Remove legend
plt.legend().remove()
# Save
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.svg', dpi=300, transparent=True)

In [52]:
# Plot interpolation results over the sample density

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', hue='Interpolation number', marker='o', s=50, palette='icefire', edgecolor='black')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='Interpolation number', marker='o', s=50, palette='icefire', edgecolor='black')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()

# plt.xlim(-1, 0.2)
# plt.ylim(-0.8, 0.6)

plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_sample_density.pdf', dpi=300)

# Save svg for reconstruction figure

# Remove axis border, labels and ticks
plt.axis('off')
# Remove legend
plt.legend().remove()
# Save
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_sample_density.svg', dpi=300, transparent=True)

In [53]:
df_exp_results_2 = df_exp_results.copy()
df_exp_results_2['Label'] = ''
for i in range(len(df_exp_results_2)):
    if df_exp_results_2['composition'].iloc[i] == 'IrO2':
        df_exp_results_2['Label'].iloc[i] = 'IrO2 (Rutile)'
    elif df_exp_results_2['composition'].iloc[i] == 'CeO2':
        df_exp_results_2['Label'].iloc[i] = 'CeO2 (Fluorite)'
    else:
        df_exp_results_2['Label'].iloc[i] = 'HEA (Spinel)'
        
# # Remove index 1
# df_exp_results_2.drop(index=1, inplace=True)

In [54]:
# Plot experimental data over the sample density

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_exp_results_2, x='ls1', y='ls2', color='black', style='Label', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_exp_results_2, x='pc1', y='pc2', color='black', style='Label', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_sample_density.pdf', dpi=300)

In [55]:
df_exp_results

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,ls1,ls2,ground_truth_crystal_type
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[0.34958064556121826, -1.4048129320144653]","[0.12149949371814728, 0.17361588776111603]","[0.0590512752532959, -1.3987634181976318]","[4.596202373504639, 4.632835388183594, 2.87167...","[[0.04129000008106232, -0.03585999831557274, 0...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.3495806,-1.4048129,Rutile
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[-0.8993464112281799, 0.10989199578762054]","[0.07034019380807877, 0.14894959330558777]","[-0.8272374272346497, 0.10075908899307251]","[8.476621627807617, 8.472660064697266, 8.42813...","[[0.5142099857330322, 0.5121899843215942, 0.50...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.8993464,0.1098920,Rutile
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[0.35640615224838257, -1.437368631362915]","[0.11025744676589966, 0.16118021309375763]","[0.2925814092159271, -1.1656876802444458]","[4.631579399108887, 4.635101795196533, 2.95615...","[[0.012000000104308128, -0.0036100000143051147...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.3564062,-1.4373686,Rutile
3,Cr0.66Mn0.66Fe0.66Ni0.5Cu0.5O4,"[[0.0], [3.48158209817484e-05], [6.78441429045...","[-0.4976503252983093, -0.5985623002052307]","[0.08616134524345398, 0.14953884482383728]","[-0.4540221095085144, -0.5289234519004822]","[5.586108207702637, 5.600020885467529, 5.57305...","[[-0.0014900000533089042, 0.005940000060945749...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.4976503,-0.5985623,Fluorite
4,Co0.33Ni0.33Zn0.33CrMnO4,"[[0.0], [2.2900278509041527e-06], [4.542134774...","[-0.6564680933952332, -0.18515054881572723]","[0.06771566718816757, 0.13834622502326965]","[-0.6105645895004272, -0.07228562980890274]","[7.366678237915039, 7.280555248260498, 7.15318...","[[0.24844999611377716, 0.25964999198913574, 0....","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.6564681,-0.1851505,Fluorite
5,Co0.33Ni0.33Cu0.33MnFeO4,"[[0.0], [2.3368893380393274e-05], [4.552564132...","[-0.724327027797699, 0.04268096759915352]","[0.05900263413786888, 0.1300533264875412]","[-0.7121967077255249, 0.07162999361753464]","[8.166360855102539, 8.155437469482422, 8.09622...","[[0.5030900239944458, 0.5071499943733215, 0.49...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.7243270,0.0426810,Fluorite
6,Co0.33Zn0.33Ni0.33CrMnO4,"[[0.0], [0.001129354815930128], [0.00220868061...","[-0.7724447846412659, 0.15924957394599915]","[0.05540238693356514, 0.12250595539808273]","[-0.7618687152862549, 0.23634099960327148]","[8.260248184204102, 8.255254745483398, 8.23208...","[[0.5202400088310242, 0.519819974899292, 0.504...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.7724448,0.1592496,Spinel
7,CrMnCo0.33Ni0.33Cu0.33O4,"[[0.0], [3.737132601600024e-06], [7.2965626713...","[-0.42907601594924927, -0.8236934542655945]","[0.09188641607761383, 0.15597108006477356]","[-0.5682460069656372, -0.7917898893356323]","[5.348183631896973, 5.3582868576049805, 5.2909...","[[0.002409999957308173, 0.00026000000070780516...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.4290760,-0.8236935,Spinel
8,CeO2,"[[0.0], [-0.0003009798820130527], [-0.00059319...","[-0.46544915437698364, -0.8539272546768188]","[0.1053207516670227, 0.17308750748634338]","[-0.448340505361557, -0.6605030298233032]","[5.3735671043396, 5.381111145019531, 5.3401913...","[[-0.0015699999639764428, 0.002929999958723783...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.4654492,-0.8539273,Spinel
9,CeO2,"[[0.0], [0.0007327766506932676], [0.0014382996...","[-0.4588150382041931, -0.8527804613113403]","[0.10431142151355743, 0.1725536286830902]","[-0.4841683804988861, -0.8360515832901001]","[5.2734527587890625, 5.291980266571045, 5.2328...","[[-0.0016299999551847577, 0.006070000119507313...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.458815

In [56]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Rutile,"[0.34804725646972656, -1.4523036479949951]","[4.567151069641113, 4.565000534057617, 2.87790...","[[0.002739999908953905, -0.01370999962091446, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.0953129,0.3480473,-1.4523036
1,0_0,sample,Rutile,"[0.5179596543312073, -1.2605674266815186]","[4.494833469390869, 4.47055721282959, 2.888001...","[[0.001769999973475933, -0.007530000060796738,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.9170761,0.5179597,-1.2605674
2,0_1,sample,Rutile,"[0.4274711012840271, -1.723741054534912]","[4.5453596115112305, 4.533892631530762, 2.8714...","[[0.011939999647438526, -0.016189999878406525,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.3335764,0.4274711,-1.7237411
3,0_2,sample,Rutile,"[0.4078691303730011, -1.6114574670791626]","[4.5486297607421875, 4.542178153991699, 2.8620...","[[0.011769999749958515, -0.021460000425577164,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.1781759,0.4078691,-1.6114575
4,0_3,sample,Rutile,"[0.3442496359348297, -1.6591724157333374]","[4.554953098297119, 4.5517258644104, 2.8566992...","[[0.003389999968931079, -0.019300000742077827,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.2322302,0.3442496,-1.6591724


In [57]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_full_latent[contour_column].min()
max_loss = df_full_latent[contour_column].max()

# Plot
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['ls1'].min()-0.1, df_full_latent['ls1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['ls2'].min()-0.1, df_full_latent['ls2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['ls1'], df_full_latent['ls2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)#, hue_order=legend_order, style_order=legend_order)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['pc1'].min()-0.1, df_full_latent['pc1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['pc2'].min()-0.1, df_full_latent['pc2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['pc1'], df_full_latent['pc2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order, style_order=legend_order)
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_loss.pdf', dpi=300)	

In [58]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Rutile,"[0.34804725646972656, -1.4523036479949951]","[4.567151069641113, 4.565000534057617, 2.87790...","[[0.002739999908953905, -0.01370999962091446, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.0953129,0.3480473,-1.4523036
1,0_0,sample,Rutile,"[0.5179596543312073, -1.2605674266815186]","[4.494833469390869, 4.47055721282959, 2.888001...","[[0.001769999973475933, -0.007530000060796738,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.9170761,0.5179597,-1.2605674
2,0_1,sample,Rutile,"[0.4274711012840271, -1.723741054534912]","[4.5453596115112305, 4.533892631530762, 2.8714...","[[0.011939999647438526, -0.016189999878406525,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.3335764,0.4274711,-1.7237411
3,0_2,sample,Rutile,"[0.4078691303730011, -1.6114574670791626]","[4.5486297607421875, 4.542178153991699, 2.8620...","[[0.011769999749958515, -0.021460000425577164,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.1781759,0.4078691,-1.6114575
4,0_3,sample,Rutile,"[0.3442496359348297, -1.6591724157333374]","[4.554953098297119, 4.5517258644104, 2.8566992...","[[0.003389999968931079, -0.019300000742077827,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.2322302,0.3442496,-1.6591724


In [59]:
# Plot all samples in 3d if latent space is 3D

# if len(df_rec['latent_space_mean'].iloc[0]) == 3:
    
#     df_crystal = df_full_latent.copy()
#     df_crystal[['ls1', 'ls2', 'ls3']] = df_crystal['latent_point'].apply(pd.Series)
    
#     fig = px.scatter_3d(df_crystal, x='ls1', y='ls2', z='ls3', color='crystalType', symbol='crystalType', hover_data=['name', 'ls1', 'ls2', 'ls3', 'reconstruction_loss', 'cell_parameters'], color_discrete_sequence=px.colors.qualitative.Dark24)
    
#     fig.update_layout(
#         scene = dict(
#             xaxis_title='LS dim 1',
#             yaxis_title='LS dim 2',
#             zaxis_title='LS dim 3',
#         ),
#         margin=dict(l=0, r=0, t=0, b=0),
#     )
    # fig.show()

# Interpolation dataset input to model

In [60]:
# Load CHILI dataset
dataset_interp = CHILI(
    root=setup_json['data']['root'],
    dataset='CHILI-Interpolation',
    graph_type=setup_json['data']['graph_type'],
)

# Load/create data splits
try:
    # Load existing data split
    dataset_interp.load_data_split(
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
except FileNotFoundError:
    # Create new data split
    dataset_interp.create_data_split(
        test_size=setup_json['data']['split']['test'],
        validation_size=setup_json['data']['split']['validation'],
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
    
    # Load data split
    dataset_interp.load_data_split(
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
    
# Dataloader
# data_loader_interp = DataLoader(dataset_interp.validation_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
# data_loader_interp = DataLoader(dataset_interp.train_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
data_loader_interp = DataLoader(dataset_interp.test_set, batch_size=setup_json['data']['batch_size'], shuffle=False)

In [61]:
# Load CHILI dataset
dataset_interp_uc = CHILI(
    root=setup_json['data']['root'],
    dataset='CHILI-Interpolation',
    graph_type='unit_cell',
)

# Load existing data split
dataset_interp_uc.load_data_split(
    split_strategy=setup_json['data']['split_strategy'],
    stratify_on=setup_json['data']['stratify_column'],
)
    
# Dataloader
# dataset_interp_uc = DataLoader(dataset_interp.validation_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
# dataset_interp_uc = DataLoader(dataset_interp.train_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
dataset_interp_uc = DataLoader(dataset_interp.test_set, batch_size=setup_json['data']['batch_size'], shuffle=False)

In [62]:
interp_data_results = {
    'name': [],
    'crystal_type': [],
    'pdf': [],
    'n_atoms': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_atoms': [],
    'cell_positions': [],
}
    

In [63]:
# Inference
model.eval()
for batch in tqdm(dataset_interp_uc, desc='Inference', disable=setup_json['disable_tqdm']):
    batch = batch.to(device)
    this_batch_size = batch.batch.amax().item() + 1
    
    try:
        n_atoms = batch.y['unit_cell_n_atoms']
    except:
        n_atoms = batch.y['n_atoms']
    
    # Normalize scattering
    pdf = batch.y['xPDF'][:,1,:].unsqueeze(-1)
    if setup_json['data']['normalize_scattering']:
        # Normalize so highest peak in each sample is 1
        pdf /= torch.amax(pdf, dim=1, keepdim=True)[0]
    
    # Composition conditioning
    composition = torch.zeros(this_batch_size, 119).to(device)
    elements_in_batch = batch.y['atomic_species'].long()
    index_counter = 0
    for i in range(this_batch_size):
        n_elements = batch.y['n_atomic_species'][i]
        composition[i, elements_in_batch[index_counter:index_counter + n_elements]] = 1
        index_counter += n_elements
    composition[:, 0] = 1 
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
        
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store names
    interp_data_results['name'].extend(batch.data_id)
    interp_data_results['crystal_type'].extend(batch.y['crystal_type'])
    
    # Store PDF
    interp_data_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store number of atoms
    interp_data_results['n_atoms'].extend(n_atoms.cpu().tolist())
    
    # Store latent representation
    interp_data_results['prior_mean'].extend(prior_mean.cpu().tolist())
    interp_data_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    interp_data_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    interp_data_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_data_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_data_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # # Create CIF files
    # for batch_index in range(this_batch_size):
    #     # Prediction
    #     try:
    #         create_cif(
    #             cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
    #             cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
    #             cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
    #             filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
    #             prediction=True,
    #             composition=data_composition_string[composition_string_index[batch_index]],
    #             simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
    #         )
    #     except:
    #         print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_interp_data_results = pd.DataFrame(interp_data_results)
if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    df_interp_data_results[['ls1', 'ls2']] = df_interp_data_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_interp_data_results[['pc1', 'pc2']] = pca.transform(np.array(df_interp_data_results['prior_mean'].values.tolist()))
    
df_interp_data_results['Step'] = 0
df_interp_data_results['Step'][df_interp_data_results['crystal_type'] == 'interpolated'] = df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated']['name'].str.split('_').str[-3].str[-1].astype(int)

df_interp_data_results.head()

,name,crystal_type,pdf,n_atoms,prior_mean,prior_log_std,z_sample,cell_parameters,cell_atoms,cell_positions,ls1,ls2,Step
0,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.0013042003847658634], [-0.00260824...",25,"[0.6717683672904968, 1.4509018659591675]","[0.07940799742937088, 0.08784561604261398]","[0.6871750354766846, 1.6407690048217773]","[3.1410269737243652, 3.1431884765625, 4.584807...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...","[[-0.0021899999119341373, 0.002300000051036477...",0.6717684,1.4509019,4
1,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [2.421927456452977e-05], [-8.280658948...",25,"[0.6724992990493774, 1.4175138473510742]","[0.079502172768116, 0.08650851249694824]","[0.6588209271430969, 1.4909647703170776]","[3.007564067840576, 3.0113160610198975, 4.3067...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...","[[-0.0008500000112690032, -0.00173000001814216...",0.6724993,1.4175138,4
2,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.0006806306191720068], [-0.00133807...",29,"[0.8264816403388977, 0.5688767433166504]","[0.06486069411039352, 0.07581686973571777]","[0.7016637921333313, 0.618384063243866]","[3.00925874710083, 2.992405891418457, 5.186837...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...","[[0.006800000090152025, -0.006539999973028898,...",0.8264816,0.5688767,3
3,NickelArsenide_Dy,NickelArsenide,"[[0.0], [-0.0010178708471357822], [-0.00201485...",32,"[0.6621779203414917, 0.11494749784469604]","[0.06433681398630142, 0.08916953951120377]","[0.6203949451446533, 0.042049191892147064]","[4.89334774017334, 4.876428604125977, 4.966224...","[1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, ...","[[-0.003470000112429261, -0.03536999970674515,...",0.6621779,0.1149475,0
4,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-1.0812695109052584e-05], [-6.2332364...",26,"[0.32539913058280945, 1.0725325345993042]","[0.0651264637708664, 0.09378927201032639]","[0.3597835898399353, 0.9529184103012085]","[2.8720173835754395, 2.896965503692627, 4.4842...","[2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 2, 1, 1, 2, 2, ...","[[0.31442999839782715, 0.6322900056838989, 0.0...",0.3253991,1.0725325,3


## Plot

In [64]:
# Plot interpolation points in latent space

if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp_data_results, x='ls1', y='ls2', color='black', marker='o', s=50)
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=50, label='NickelArsenide')
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=50, label='CadmiumIodide')
    # sns.histplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', palette='viridis', binwidth=0.02, alpha=1, kde=True)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', marker='.', palette='viridis', s=150, edgecolor=None, alpha=0.75)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp_data_results, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=75, label='NickelArsenide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=75, label='CadmiumIodide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='pc1', y='pc2', hue='Step', marker='.', palette='viridis', s=75, edgecolor=None)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
# plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_dataset.pdf', dpi=300)


In [65]:
# Plot interpolation points in latent space (# of atoms)

if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)

    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=50, label='NickelArsenide')
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=50, label='CadmiumIodide')
    # # sns.histplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', palette='viridis', binwidth=0.02, alpha=1, kde=True)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='n_atoms', marker='.', palette='viridis', s=150, edgecolor=None, alpha=0.75)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title='# of atoms')
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)

    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=75, label='NickelArsenide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=75, label='CadmiumIodide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='pc1', y='pc2', hue='Step', marker='.', palette='viridis', s=75, edgecolor=None)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
# plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_dataset_nAtoms.pdf', dpi=300)

In [92]:
# Read in the CHILI-3K dataset
dataset_chili_3k = CHILI(
    root=setup_json['data']['root'],
    dataset='CHILI-3K',
    graph_type=setup_json['data']['graph_type'],
)

# Load existing data split
dataset_chili_3k.load_data_split(
    split_strategy=setup_json['data']['split_strategy'],
    stratify_on=setup_json['data']['stratify_column'],
)

# Dataloader
train_loader_chili_3k = DataLoader(dataset_chili_3k, batch_size=setup_json['data']['batch_size'], shuffle=False)


In [106]:
train_pdf_df = pd.DataFrame(columns=['name', 'crystalType', 'metal', 'np_size', 'pdf'])
name_list = []
crystal_type_list = []
metal_list = []
np_size_list = []
pdf_list = []
for batch in tqdm(train_loader_chili_3k, desc='Processing CHILI-3K train set', disable=setup_json['disable_tqdm']):
    batch = batch.to(device)
    
    # Get names
    name_list.extend(batch.data_id)
    
    # Get crystal types
    crystal_type_list.extend(batch.y['crystal_type'])
    
    # Get metal
    metal_list.extend(batch.y['atomic_species'].cpu().tolist())

    # Get np size
    np_size_list.extend(batch.y['np_size'].cpu().tolist())
    
    # Get PDFs
    pdf_list.extend(batch.y['xPDF'][:,1,:].cpu().tolist())

# Remove oxygens from metal list
metal_list = metal_list[1::2]

train_pdf_df['name'] = name_list
train_pdf_df['crystalType'] = crystal_type_list
train_pdf_df['metal'] = metal_list
train_pdf_df['np_size'] = np_size_list
train_pdf_df['pdf'] = pdf_list
train_pdf_df['rounded_size'] = np.round(np_size_list, -1).astype(int)

In [107]:
train_pdf_df.head()

,name,crystalType,metal,np_size,pdf,rounded_size
0,CadmiumChloride_HfO2,CadmiumChloride,72.0000000,10.2215290,"[0.0, -0.0002874378114938736, -0.0008905497961...",10
1,CadmiumChloride_HfO2,CadmiumChloride,72.0000000,23.4750366,"[0.0, -0.0008902198751457036, -0.0019239876419...",20
2,CadmiumChloride_HfO2,CadmiumChloride,72.0000000,33.5846901,"[0.0, -0.0010275871027261019, -0.0021662730723...",30
3,CadmiumChloride_HfO2,CadmiumChloride,72.0000000,43.7684822,"[0.0, -0.0006874905084259808, -0.0015118262963...",40
4,CadmiumChloride_HfO2,CadmiumChloride,72.0000000,53.5865707,"[0.0, -0.0003250766603741795, -0.0008149503846...",50


In [108]:
# Plot the first 10 Å of experimental sample and references
exp_index = 1

ref_structures = ['Spinel', 'Rutile']
ref_metals = [77,77]
ref_sizes = [10, 10]
ref_indeces = [0, 0]

n_refs = len(ref_structures)

plt.figure(figsize=(8, 6))
plt.plot(np.arange(0,10,0.01), np.array(df_exp_results['pdf'].iloc[exp_index])[:1000]+2, label=df_exp_results['composition'].iloc[exp_index], c='k', lw=3)

for i, (ref_index, ref_structure, ref_metal, ref_size) in enumerate(zip(ref_indeces, ref_structures, ref_metals, ref_sizes)):
    norm_pdf = np.array(train_pdf_df[(train_pdf_df['crystalType'] == ref_structure) & (train_pdf_df['metal'] == ref_metal) & (train_pdf_df['rounded_size'] == ref_size)]['pdf'].iloc[ref_index])[:1000]
    norm_pdf /= np.amax(norm_pdf)
    plt.plot(np.arange(0,10,0.01), norm_pdf+i, label=ref_structure, linestyle='--')

plt.xlabel('r [Å]', fontsize=14, fontweight='bold')
plt.ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
plt.legend()
plt.title('First 10 Å of experimental samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_first_10A.pdf', dpi=300)

In [135]:
# Plot the first 10 Å of experimental sample and references
exp_index = 1


ref_structures = ['Spinel', 'Rutile']
ref_metals = [79,77]
ref_sizes = [50, 50]
ref_indeces = [0, 0]

n_refs = len(ref_structures)

fig, ax = plt.subplots(nrows=n_refs, figsize=(8,6), sharex=True)

for i, (ref_index, ref_structure, ref_metal, ref_size) in enumerate(zip(ref_indeces, ref_structures, ref_metals, ref_sizes)):
    ax[i].plot(np.arange(0,10,0.01), np.array(df_exp_results['pdf'].iloc[exp_index])[:1000], label=df_exp_results['composition'].iloc[exp_index], c='k', lw=3)

    norm_pdf = np.array(train_pdf_df[(train_pdf_df['crystalType'] == ref_structure) & (train_pdf_df['metal'] == ref_metal) & (train_pdf_df['rounded_size'] == ref_size)]['pdf'].iloc[ref_index])[:1000]
    norm_pdf /= np.amax(norm_pdf)
    ax[i].plot(np.arange(0,10,0.01), norm_pdf, label=ref_structure, linestyle='-', lw=2, c=sns.color_palette('colorblind')[i])

    ax[i].set_ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
    ax[i].legend()
plt.xlabel('r [Å]', fontsize=14, fontweight='bold')
plt.xlim(1,7)
# plt.ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
# plt.legend()
# plt.title('First 10 Å of experimental samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(hspace=0)
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_first_10A.pdf', dpi=300)